# Smart City-project van Nova Stad

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, lit, udf, pandas_udf, PandasUDFType
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder, Imputer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import os
import json
from PIL import Image

from pyspark.sql.functions import input_file_name
import time

In [0]:
%pip install ultralytics mlflow

# Data inladen

In [0]:
img_path = "bdd/images/train"
json_label_path = "bdd/json_labels/train"

# haal image namen zonder .jpg
image_ids = set([f.replace(".jpg", "") for f in os.listdir(img_path) if f.endswith(".jpg")])

# labels ophalen
labels = [f for f in os.listdir(json_label_path) if f.endswith(".json")]

# check matches
matched = [f for f in labels if f.replace(".json", "") in image_ids]

print("Aantal images", len(image_ids))
print("Aantal labels", len(labels))
print("Matches", len(matched))

### label verwijderen

In dit code gedeelte checken we matching labels en images. Niet gekoppelde items worden verwijderd.

In [0]:
matched_set = set([f.replace(".json", "") for f in matched])

In [0]:
for f in labels:
    base = f.replace(".json", "")
    
    if base not in matched_set:
        os.remove(os.path.join(json_label_path, f))

print("Alleen matched labels behouden ")

In [0]:
remaining_labels = [f for f in os.listdir(json_label_path) if f.endswith(".json")]

print("Overgebleven labels", len(remaining_labels))

### Images verwijderen

In [0]:
image_files = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
label_ids = set(os.path.splitext(f)[0] for f in os.listdir(json_label_path) if f.endswith(".json"))

delete_images = [f for f in image_files if os.path.splitext(f)[0] not in label_ids]


In [0]:
for f in delete_images:
    os.remove(os.path.join(img_path, f))

print("Images opgeschoond")

In [0]:
remaining_images = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
remaining_labels = [f for f in os.listdir(json_label_path) if f.endswith(".json")]

print("Images", len(remaining_images))
print("Labels", len(remaining_labels))

## Verdeel images en labels naar val map

In [0]:
import os
import shutil
import random

img_path = "bdd/images/train"
val_img = "bdd/images/val"
json_label_path = "bdd/json_labels/train"
val_label = "bdd/json_labels/val"

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_label, exist_ok=True)

#  Verwijder alle val images
#for f in os.listdir(val_img):
    #if f.endswith(".jpg"):
        #os.remove(os.path.join(val_img, f))

# Verwijder alle val labels
#for f in os.listdir(val_label):
    #if f.endswith(".json"):
        #os.remove(os.path.join(val_label, f))

# Kies alleen train images die ook een label hebben
image_files = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
label_ids = {os.path.splitext(f)[0] for f in os.listdir(json_label_path) if f.endswith(".json")}


matched_images = [f for f in image_files if os.path.splitext(f)[0] in label_ids]

print("Beschikbare matched train images", len(matched_images))

# Selecteer 500 images voor val
#random.seed(42)
#val_selection = random.sample(matched_images, 500)

# Verplaats image + label naar val
#for img_file in val_selection:
    #base = os.path.splitext(img_file)[0]
    #label_file = base + ".json"

    #shutil.move(os.path.join(img_path, img_file), os.path.join(val_img, img_file))
    #shutil.move(os.path.join(json_label_path, label_file), os.path.join(val_label, label_file))

#print("Klaar")

### Check aantal files

In [0]:
def count_files(path, ext):
    return len([f for f in os.listdir(path) if f.endswith(ext)])

print("Train images:", count_files(img_path, ".jpg"))
print("Train labels:", count_files(json_label_path, ".json"))
print("Val images:", count_files(val_img, ".jpg"))
print("Val labels:", count_files(val_label, ".json"))

## Data pipeline

In [0]:
df_traffic = spark.read.csv("/FileStore/tables/metro_interstate_traffic_volume_train.csv", header=True, inferSchema=True)

In [0]:
df = spark.read.csv(
    "/Volumes/main/default/datasets/metro_interstate_traffic_volume_train.csv",
    header=True,
    inferSchema=True
)



In [0]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, to_timestamp, year, month, dayofmonth, hour, dayofweek
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# Pad
train_path = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
test_path  = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"
# functie voor schoonmaken
def clean_df(df):

    # 1. duplicaten verwijderen
    df = df.dropDuplicates()

    # 2. datum omzetten
    df = df.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))

    # 3. string kolommen opschonen
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(c, regexp_replace(col(c), r"\s+", " "))

    # 4. numerieke kolommen bepalen
    numeric_types = ["int", "bigint", "double", "float", "long", "short", "decimal"]
    numeric_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() in numeric_types]

    # 5. missende numerieke waardes vullen (mediaan)
    for c in numeric_cols:
        median_value = df.approxQuantile(c, [0.5], 0.01)[0]
        df = df.fillna({c: median_value})

    # 6. missende string waardes vullen
    for c in string_cols:
        df = df.fillna({c: "Unknown"})

    # 7. datum features maken
    df = (
        df
        .withColumn("year", year(col("date_time")))
        .withColumn("month", month(col("date_time")))
        .withColumn("day", dayofmonth(col("date_time")))
        .withColumn("hour", hour(col("date_time")))
        .withColumn("dayofweek", dayofweek(col("date_time")))
    )

    return df


# Alles in 1 lijn uitvoeren (dan wordt neiwue df een keer opgeslagen)
df_train = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(train_path)
)

df_test = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(test_path)
)

# Resultaat bekijken
display(df_train.limit(5))
display(df_test.limit(5))

# YOLO

### Bestand converten

ChatGPT 5.2, Prompt 2: JSON naar YOLO TXT converteren, https://chatgpt.com/share/6a0339c0-52ac-8390-bb0e-3daf97c80b68


In [0]:
# CLASS MAPPING

# classes = {
#     "car": 0,
#     "bus": 1,
#     "truck": 2,
#     "person": 3,
#     "traffic sign": 4,
#     "traffic light": 5,
#     "bike": 6,
#     "motor": 7
# }


# CONVERTER FUNCTIE

# def convert_to_yolo(img_path, label_path, output_dir):

#     os.makedirs(output_dir, exist_ok=True)

#     for json_file in os.listdir(label_path):

#         # alleen json files
#         if not json_file.endswith(".json"):
#             continue

#         json_path = os.path.join(label_path, json_file)

#         # json openen
#         with open(json_path, "r") as f:
#             data = json.load(f)

#         # image naam ophalen
#         image_name = data["name"] + ".jpg"

#         image_path = os.path.join(img_path, image_name)

#         # check image bestaat
#         if not os.path.exists(image_path):
#             print(f"Image niet gevonden: {image_name}")
#             continue

#         # image openen
#         img = Image.open(image_path)

#         img_width, img_height = img.size

#         yolo_lines = []

#         frames = data.get("frames", [])

#         for frame in frames:

#             objects = frame.get("objects", [])

#             for obj in objects:

#                 category = obj.get("category")

#                 # alleen gewenste classes
#                 if category not in classes:
#                     continue

#                 # alleen box2d objecten
#                 if "box2d" not in obj:
#                     continue

#                 box = obj["box2d"]

#                 x1 = box["x1"]
#                 y1 = box["y1"]
#                 x2 = box["x2"]
#                 y2 = box["y2"]


#                 # YOLO CONVERSIE

#                 x_center = (x1 + x2) / 2
#                 y_center = (y1 + y2) / 2

#                 width = x2 - x1
#                 height = y2 - y1

#                 # NORMALISEREN

#                 x_center /= img_width
#                 y_center /= img_height

#                 width /= img_width
#                 height /= img_height

#                 class_id = classes[category]

#                 line = f"{class_id} {x_center} {y_center} {width} {height}"

#                 yolo_lines.append(line)

#         # txt file opslaan
#         txt_name = os.path.splitext(json_file)[0] + ".txt"

#         txt_path = os.path.join(output_dir, txt_name)

#         with open(txt_path, "w") as f:
#             f.write("\n".join(yolo_lines))

#         print(f"Geconverteerd {json_file}")

#     print(f"Klaar {output_dir}")


# TRAIN MAP CONVERTEREN

# convert_to_yolo(
#     img_path="bdd/images/train",
#     label_path="bdd/json_labels/train",
#     output_dir="bdd/labels/train"
# )

# VAL MAP CONVERTEREN

# convert_to_yolo(
#     img_path="bdd/images/val",
#     label_path="bdd/json_labels/val",
#     output_dir="bdd/labels/val"
# )

In [0]:
labels = "bdd/labels/train"


print("Train images:", count_files(img_path, ".jpg"))
print("Train labels:", count_files(labels, ".txt"))



print("Val images:", count_files(val_img, ".jpg"))
print("Val labels:", count_files(val_label, ".json"))

### Model trainen

In [0]:

## laad image data

## Pas je eigen path aan naar de juiste locatie van de train images!!!!!
images_df = spark.read.format("binaryFile") \
    .load("file:/Workspace/Users/jerniceryarmy@gmail.com/MLops-City/bdd/images/train/*.jpg")

print("Aantal train images:", images_df.count())

images_df.show(5)

In [0]:
## laad label data
labels_df = spark.read.text("file:/Workspace/Users/jerniceryarmy@gmail.com/MLops-City/bdd/labels/train/*.txt")

print("Aantal train labels:", labels_df.count())

labels_df.show(5, truncate=False)

In [0]:
## modeel trainen
import mlflow
# set experiment
mlflow.set_experiment("/Shared/YOLOv8_Traffic_Detection")

import os
import time
import mlflow
from ultralytics import YOLO

# mlflow run starten
with mlflow.start_run(run_name="yolov8n_traffic"):
    # Hyperparameters loggen

    mlflow.log_param("model", "yolov8n")
    mlflow.log_param("epochs", 100)
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("batch", 20)
    mlflow.log_param("optimizer", "AdamW")
    mlflow.log_param("edge_target", "Jetson Nano")

    # YOLO model laden
    model = YOLO("yolov8n.pt")

    # Training starten
    start_time = time.time()

    yolo = model.train(
        data="data.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        optimizer="AdamW",
        amp=True,
        patience=10,
        cache=True,
        workers=4,
        project="traffic_yolo",
        name="yolov8n_edge"
    )

    training_time = time.time() - start_time
    print(f"Training duurde {training_time} seconden")

    # Validatie uitvoeren
    metrics = model.val()

    # Metrics loggen
    mlflow.log_metric("training_time_seconds", training_time)

    # mAP50
    mlflow.log_metric(
        "mAP50",
        float(metrics.box.map50)
    )

    # mAP50-95
    mlflow.log_metric(
        "mAP50_95",
        float(metrics.box.map)
    )

    # precision
    mlflow.log_metric(
        "precision",
        float(metrics.box.mp)
    )

    # recall
    mlflow.log_metric(
        "recall",
        float(metrics.box.mr)
    )

    # Modelgrootte bepalen
    best_model_path = "traffic_yolo/yolov8n_edge/weights/best.pt"

    size_mb = os.path.getsize(best_model_path) / (1024 * 1024)

    mlflow.log_metric("model_size_mb", size_mb)

    # Model loggen in MLflow
    mlflow.log_artifact(best_model_path)

    print("Training voltooid")